# JSON Data Quality Audit
Analyze JSON structure, depth, and key consistency across nested data.

In [9]:
import json
import os
import pandas as pd
from typing import Any, Dict, List
print('JSON Audit Setup Complete.')

JSON Audit Setup Complete.


## Section 1: Structural Audit
Validating schema consistency and nesting depth.

In [10]:
def get_json_depth(d, level=1):
    if not isinstance(d, (dict, list)) or not d:
        return level
    if isinstance(d, dict):
        return max(get_json_depth(v, level + 1) for v in d.values())
    return max(get_json_depth(item, level + 1) for item in d)

def check_key_consistency(data_list: List[Dict]):
    if not data_list or not isinstance(data_list[0], dict):
        return None
    all_keys = set()
    for d in data_list: all_keys.update(d.keys())
    
    report = []
    for key in all_keys:
        missing = sum(1 for d in data_list if key not in d)
        if missing > 0:
            report.append({'Key': key, 'Missing Count': missing, 'Presence %': f'{(1 - missing/len(data_list))*100:.1f}%'})
    return pd.DataFrame(report)

## Section 2: Content Analysis
Detecting nulls and type mismatches in flat or nested components.

In [11]:
def audit_json_content(data):
    if isinstance(data, list):
        df = pd.json_normalize(data)
        return df.isnull().sum().to_frame('Null Count')
    return "Data is not a list of objects; flat analysis skipped."

## Section 3: Audit Execution
Main report generator for JSON files.

In [12]:
def full_audit_json(file_path: str):
    print(f"--- AUDIT REPORT: {os.path.basename(file_path)} ---")
    with open(file_path, 'r') as f:
        data = json.load(f)
    
    print(f"Max Nesting Depth: {get_json_depth(data)}")
    
    if isinstance(data, list):
        print(f"Structure: List of {len(data)} items")
        consistency = check_key_consistency(data)
        if consistency is not None and not consistency.empty:
            print("\nKey Inconsistency (Missing keys in some objects):")
            display(consistency)
            
        content = audit_json_content(data)
        if isinstance(content, pd.DataFrame):
            print("\nNull Value Density (Normalized):")
            display(content)
    else:
        print("Structure: Single Object Root")

test_dir = r'd:\01.CodingProjects\DataQuality\test_files'
for f in os.listdir(test_dir):
    if f.endswith('.json'):
        full_audit_json(os.path.join(test_dir, f))

--- AUDIT REPORT: events_messy.json ---
Max Nesting Depth: 3
Structure: List of 4 items

Key Inconsistency (Missing keys in some objects):


,Key,Missing Count,Presence %
0,timestamp,2,50.0%
1,extra_key,3,25.0%



Null Value Density (Normalized):


,Null Count
id,0
event,0
timestamp,2
extra_key,3


--- AUDIT REPORT: products_clean.json ---
Max Nesting Depth: 3
Structure: List of 3 items

Null Value Density (Normalized):


,Null Count
id,0
name,0
price,0


--- AUDIT REPORT: sensor_data.json ---
Max Nesting Depth: 6
Structure: Single Object Root
